# Лабораторная работа №7 Цвиль Павел ФИТ-231


# Задание 1
a) Перепишите функции buildParseTree и evaluate из лекционных материалов так,
чтобы они работали с логическими выражениями, то есть с операторами «и», «или»,
«не» и операндами «истина» и «ложь». Помните, что «не» – унарный оператор.

b) Реализуйте функцию printexp, которая будет принимать дерево синтаксического
разбора и возвращать строку соответствующего ему логического выражения.

In [1]:
class BinaryTree:
    def __init__(self, rootObj):
        self.key = rootObj; self.leftChild = None; self.rightChild = None
    def insertLeft(self, newNode): self.leftChild = newNode if isinstance(newNode, BinaryTree) else BinaryTree(newNode)
    def insertRight(self, newNode): self.rightChild = newNode if isinstance(newNode, BinaryTree) else BinaryTree(newNode)
    def getRightChild(self): return self.rightChild
    def getLeftChild(self): return self.leftChild
    def getRootVal(self): return self.key


def normalize_token(token):
    return {"истина": True, "ложь": False, "true": True, "false": False, "и": "and", "или": "or", "не": "not", "and": "and", "or": "or", "not": "not"}.get(token.lower(), token)


def buildParseTree(expr):
    tokens = expr.replace("(", " ( ").replace(")", " ) ").split(); pos = 0
    def parse():
        nonlocal pos
        token = tokens[pos]; pos += 1
        if token == "(":
            if normalize_token(tokens[pos]) == "not":
                pos += 1; child = parse(); pos += 1
                node = BinaryTree("not"); node.insertLeft(child); return node
            left = parse(); op = normalize_token(tokens[pos]); pos += 1; right = parse(); pos += 1
            node = BinaryTree(op); node.insertLeft(left); node.insertRight(right); return node
        return BinaryTree(normalize_token(token))
    return parse()


def evaluate(tree):
    val, left, right = tree.getRootVal(), tree.getLeftChild(), tree.getRightChild()
    if left is None and right is None: return val
    if val == "not": return not evaluate(left)
    if val == "and": return evaluate(left) and evaluate(right)
    if val == "or": return evaluate(left) or evaluate(right)


def printexp(tree):
    val, left, right = tree.getRootVal(), tree.getLeftChild(), tree.getRightChild()
    if left is None and right is None: return "истина" if val is True else "ложь" if val is False else str(val)
    if val == "not": return f"(не {printexp(left)})"
    return f"({printexp(left)} {'и' if val == 'and' else 'или'} {printexp(right)})"

expr = "( ( не истина ) и ( ложь или истина ) )"
tree = buildParseTree(expr)
print(printexp(tree), evaluate(tree))

((не истина) и (ложь или истина)) False


# Задание 2

a) Реализуйте функцию has_no_duplicates, которая принимает в качестве аргумента
двоичное дерево поиска и возвращает «истину» если в нем нет равных ключей и
«ложь» в противном случае.

b) Измените реализацию двоичного дерева поиска из лекционных материалов так,
чтобы оно правильно работало с дубликатами ключей. Т.е., если ключ в дереве уже
присутствует, то новое значение должно заменить старое, вместо того, чтобы
добавлять новый узел с тем же ключом. Проверьте правильность вашего решения
функцией has_no_duplicates.


In [2]:
class TreeNode:
    def __init__(self, key, val):
        self.key = key; self.payload = val; self.left = None; self.right = None


class BinarySearchTree:
    def __init__(self): self.root = None
    def put(self, key, val):
        def insert(node, key, val):
            if node is None: return TreeNode(key, val)
            if key == node.key: node.payload = val
            elif key < node.key: node.left = insert(node.left, key, val)
            else: node.right = insert(node.right, key, val)
            return node
        self.root = insert(self.root, key, val)


def has_no_duplicates(node, seen=None):
    if seen is None: seen = set()
    if node is None: return True
    if node.key in seen: return False
    seen.add(node.key)
    return has_no_duplicates(node.left, seen) and has_no_duplicates(node.right, seen)

bst = BinarySearchTree()
for k, v in [(5,"a"),(3,"b"),(7,"c"),(3,"new")]:
    bst.put(k, v)
print(has_no_duplicates(bst.root), bst.root.left.payload)

True new


# Задание 3

Суть игры «Животные» заключается в следующем: программа пытается угадать
задуманное пользователем животное и при этом самообучается, то есть постепенно
совершенствуется. Вся необходимая информация хранится в бинарном дереве, каждая
вершина которого содержит вопрос, предполагающий ответ «да» или «нет».
Последовательный выбор одного из этих ответов ведет программу вниз по
соответствующим ветвям до терминальной вершины, где и находится название животного.
Если программа совершает ошибку, она просит пользователя ввести уточняющий вопрос,
который позволил бы ей прийти к верному решению, затем добавляет его в новую
внутреннюю вершину и создает для нее свои листья.
Предположим, текущая база знаний программы выглядит так, как на рисунке, и
пользователь загадал змею. В таблице приводятся список вопросов, которые задает
программа, и полученные ответы пользователя

In [3]:

import heapq


class AnimalTree:
    def __init__(self, text, yes_node=None, no_node=None):
        self.text = text
        self.yes_node = yes_node
        self.no_node = no_node

    def is_leaf(self):
        return self.yes_node is None and self.no_node is None

    def learn(self, correct_animal, question, answer_for_correct):
        old_guess = self.text
        self.text = question
        if answer_for_correct == "да":
            self.yes_node = AnimalTree(correct_animal)
            self.no_node = AnimalTree(old_guess)
        else:
            self.yes_node = AnimalTree(old_guess)
            self.no_node = AnimalTree(correct_animal)

    def play_scripted(self, answers, correct_animal=None, new_question=None, answer_for_correct=None):
        current = self
        route = []
        answer_iter = iter(answers)

        while not current.is_leaf():
            answer = next(answer_iter)
            route.append((current.text, answer))
            current = current.yes_node if answer == "да" else current.no_node

        guess = current.text
        if correct_animal is not None and correct_animal != guess:
            current.learn(correct_animal, new_question, answer_for_correct)
            return route, guess, f"Не угадали. Добавлено животное: {correct_animal}"
        return route, guess, "Угадали"


def create_initial_tree():
    return AnimalTree(
        "Оно живет на суше?",
        AnimalTree("Это большое животное?", AnimalTree("слон"), AnimalTree("кот")),
        AnimalTree("акула"),
    )


animal_tree = create_initial_tree()
route, guess, result = animal_tree.play_scripted(
    answers=["да", "нет"],
    correct_animal="змея",
    new_question="У него нет ног?",
    answer_for_correct="да",
)
print("Путь вопросов:", route)
print("Первое предположение:", guess)
print(result)

route, guess, result = animal_tree.play_scripted(answers=["да", "нет", "да"])
print("После обучения путь:", route)
print("Новое предположение:", guess)
print(result)


class BinaryHeapLimited:
    def __init__(self, max_size):
        self.max_size = max_size
        self.heap = []

    def push(self, priority, item):
        heapq.heappush(self.heap, (priority, item))
        dropped = None
        if len(self.heap) > self.max_size:
            dropped = heapq.heappop(self.heap)
        return dropped

    def items(self):
        return sorted(self.heap, reverse=True)


limited = BinaryHeapLimited(3)
for priority in [5, 1, 9, 3, 7]:
    dropped = limited.push(priority, f"item{priority}")
    print(f"Добавили {priority}, отброшено: {dropped}, куча: {limited.items()}")


class MaxBinaryHeap:
    def __init__(self):
        self.heap = []

    def insert(self, item):
        heapq.heappush(self.heap, -item)

    def delMax(self):
        return -heapq.heappop(self.heap)


class PriorityQueue:
    def __init__(self):
        self.heap = []

    def enqueue(self, item, priority):
        heapq.heappush(self.heap, (priority, item))

    def dequeue(self):
        return heapq.heappop(self.heap)[1]


mh = MaxBinaryHeap()
for value in [1, 5, 3]:
    mh.insert(value)
print("Max heap вернул:", mh.delMax())

pq = PriorityQueue()
pq.enqueue("low", 5)
pq.enqueue("high", 1)
print("PriorityQueue вернула:", pq.dequeue())


Путь вопросов: [('Оно живет на суше?', 'да'), ('Это большое животное?', 'нет')]
Первое предположение: кот
Не угадали. Добавлено животное: змея
После обучения путь: [('Оно живет на суше?', 'да'), ('Это большое животное?', 'нет'), ('У него нет ног?', 'да')]
Новое предположение: змея
Угадали
Добавили 5, отброшено: None, куча: [(5, 'item5')]
Добавили 1, отброшено: None, куча: [(5, 'item5'), (1, 'item1')]
Добавили 9, отброшено: None, куча: [(9, 'item9'), (5, 'item5'), (1, 'item1')]
Добавили 3, отброшено: (1, 'item1'), куча: [(9, 'item9'), (5, 'item5'), (3, 'item3')]
Добавили 7, отброшено: (3, 'item3'), куча: [(9, 'item9'), (7, 'item7'), (5, 'item5')]
Max heap вернул: 5
PriorityQueue вернула: high
